## Configuration and Reset Utilities
This cell sets up project paths, loads the master dataset, and defines utility functions for resetting checkpoints, embeddings, and results. It also provides a quick start guide for managing the pipeline state. Run this cell before starting any processing to ensure all paths and utilities are initialized.

In [1]:
# =============================================================================
# CONFIGURATION & RESET UTILITIES — ITERATION 5
# =============================================================================

import os
import json
import glob
import shutil
from pathlib import Path
import pandas as pd

def find_project_root(start_path: Path, max_levels: int = 10) -> Path:
    """Search upward from start_path for directory containing 'Master Dataset 34k'."""
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Master Dataset 34k').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not find project root with 'Master Dataset 34k' within {max_levels} levels. "
        f"Set THESIS_BASE_DIR environment variable or ensure the dataset folder exists."
    )

env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base).expanduser().resolve()
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'master_dataset': (BASE_DIR / 'Master Dataset 34k').resolve(),
    'embeddings': (BASE_DIR / 'Embeddings' / '_iteration_5').resolve(),
    'results': (BASE_DIR / 'Results' / '_iteration_5').resolve(),
}

if not PATHS['master_dataset'].exists():
    raise FileNotFoundError(f"Master dataset folder not found: {PATHS['master_dataset']}")
PATHS['embeddings'].mkdir(parents=True, exist_ok=True)
PATHS['results'].mkdir(parents=True, exist_ok=True)

DATA_DIR = str((PATHS['master_dataset'] / 'By_SL_Country').resolve())
EMBEDDINGS_BASE_DIR = str(PATHS['embeddings'])
RESULTS_DIR = str(PATHS['results'])
CHECKPOINT_FILE = str(PATHS['results'] / 'checkpoint_iteration_5.json')

# Language code mapping (consistent with Iteration 0)
LANGUAGE_CODE_MAP = {
    'English': 'EN', 'German': 'DE', 'Swedish': 'SV',
    'Dutch': 'NL', 'Hungarian': 'HU', 'Unknown': 'UN',
}

# Load Master Dataset (pre-processed in Iteration 0)
master_df = pd.read_json(PATHS['master_dataset'] / 'master_df.json')
print(f"Loaded master_df: {len(master_df):,} records, {master_df.shape[1]} columns")

# Dataset files (from Iteration 0 outputs)
DATASET_FILES = {
    'english_manual': str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_English_manual.json'),
    'german_manual': str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Germany_manual.json'),
    'swedish_manual': str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Sweden_manual.json'),
    'dutch_manual': str(PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Netherlands_manual.json'),
}
DATASET_FILES = {k: v for k, v in DATASET_FILES.items() if os.path.exists(v)}
if not DATASET_FILES:
    raise FileNotFoundError(
        "No iteration-0 JSON dataset files were found in 'Master Dataset 34k/By_SL_Country'."
    )

print(f"Dataset files available: {list(DATASET_FILES.keys())}")
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"EMBEDDINGS_BASE_DIR: {EMBEDDINGS_BASE_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

def reset_checkpoint():
    """Delete checkpoint to restart from beginning"""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared - will restart from beginning")
    else:
        print("No checkpoint found - already clean")

def view_progress():
    """View current progress"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
        processed = data['processed']
        print(f"\n{'='*60}")
        print(f"ITERATION 5 PROGRESS: {len(processed)} combinations processed")
        print(f"{'='*60}")
        print("\nCompleted:")
        for item in processed:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                model, dataset = item[0], item[1]
                print(f"  {model:45s} → {dataset}")
            else:
                print(f"  {item}")
    else:
        print("No checkpoint found - no progress yet")

def reset_embeddings():
    """Delete all generated embeddings for iteration 5"""
    if os.path.exists(EMBEDDINGS_BASE_DIR):
        response = input(f"Delete ALL embeddings in {EMBEDDINGS_BASE_DIR}? (yes/no): ")
        if response.lower() == 'yes':
            shutil.rmtree(EMBEDDINGS_BASE_DIR)
            os.makedirs(EMBEDDINGS_BASE_DIR)
            print("All iteration 5 embeddings deleted")
        else:
            print("Cancelled")
    else:
        print("Embeddings directory doesn't exist")

def reset_results():
    """Delete all results for iteration 5"""
    if os.path.exists(RESULTS_DIR):
        response = input(f"Delete ALL results in {RESULTS_DIR}? (yes/no): ")
        if response.lower() == 'yes':
            for file in glob.glob(os.path.join(RESULTS_DIR, '*')):
                os.remove(file)
            print("All iteration 5 results deleted")
        else:
            print("Cancelled")
    else:
        print("Results directory doesn't exist")

def full_reset():
    """Complete reset - checkpoint, embeddings, and results"""
    print("\n" + "="*60)
    print("FULL RESET WARNING - ITERATION 5")
    print("="*60)
    response = input("This will delete EVERYTHING (checkpoint, embeddings, results). Continue? (yes/no): ")
    if response.lower() == 'yes':
        reset_checkpoint()
        if os.path.exists(EMBEDDINGS_BASE_DIR):
            shutil.rmtree(EMBEDDINGS_BASE_DIR)
            os.makedirs(EMBEDDINGS_BASE_DIR)
            print("Embeddings deleted")
        if os.path.exists(RESULTS_DIR):
            for file in glob.glob(os.path.join(RESULTS_DIR, '*')):
                os.remove(file)
            print("Results deleted")
        print("\nFull reset complete - ready for fresh start")
    else:
        print("Cancelled")

def remove_specific_model(model_key):
    """Remove specific model from checkpoint to reprocess it"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
        processed = data['processed']

        new_processed = [p for p in processed if not (isinstance(p, (list, tuple)) and p[0] == model_key)]

        if len(new_processed) < len(processed):
            data['processed'] = new_processed
            with open(CHECKPOINT_FILE, 'w') as f:
                json.dump(data, f)
            print(f"Removed {model_key} from checkpoint - will be reprocessed")
        else:
            print(f"{model_key} not found in checkpoint")
    else:
        print("No checkpoint found")

def complete_rerun():
    """Simple one-command complete restart for iteration 5"""
    print("\n" + "="*60)
    print("COMPLETE RERUN - ITERATION 5")
    print("="*60)
    print("This will:")
    print("  1. Clear checkpoint file")
    print("  2. Delete all embeddings")
    print("  3. Delete all results")
    print("  4. Start fresh from beginning")
    print("="*60)
    response = input("\nProceed with complete rerun? (yes/no): ")
    if response.lower() == 'yes':
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            print("Checkpoint cleared")

        if os.path.exists(EMBEDDINGS_BASE_DIR):
            shutil.rmtree(EMBEDDINGS_BASE_DIR)
            os.makedirs(EMBEDDINGS_BASE_DIR)
            print("Embeddings cleared")

        if os.path.exists(RESULTS_DIR):
            for file in glob.glob(os.path.join(RESULTS_DIR, '*')):
                os.remove(file)
            print("Results cleared")

        print("\nReady for complete rerun!")
        print("Run the main processing cell to start fresh.")
    else:
        print("Cancelled")

# ============================================================
# QUICK START GUIDE
# ============================================================
print("""
============================================================
ITERATION 5 - RESET UTILITIES
============================================================
Available functions:
  view_progress() - See what's been processed
  reset_checkpoint() - Clear checkpoint to restart
  reset_embeddings() - Delete all embeddings
  reset_results() - Delete all results
  full_reset() - Complete reset (all of above)
  complete_rerun() - One-command complete restart
  remove_specific_model('model_name') - Remove specific model

Run the next cell to start processing.
============================================================
""")


Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
Loaded master_df: 34,576 records, 36 columns
Dataset files available: ['english_manual', 'german_manual', 'swedish_manual', 'dutch_manual']
BASE_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
DATA_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/By_SL_Country
EMBEDDINGS_BASE_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Embeddings/_iteration_5
RESULTS_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_5

ITERATION 5 - RESET UTILITIES
Available functions:
  view_progress() - See what's been processed
  reset_checkpoint() - Clear checkpoint to restart
  reset_embeddings() - Delete all embeddings
  reset_results() - Delete all results
  full_reset() - Complete reset (all of above)
  complete_rerun() - One-command complete restart
  remove_specific_model('model_name') - Remove specific model

Run the next cell to

## (Optional) Complete Rerun Utility
This cell allows you to perform a complete reset of the pipeline, clearing all checkpoints, embeddings, and results. Uncomment and run the code if you want to start the entire iteration from scratch.

In [2]:
# complete_rerun()

## Aggressive Process Safety Targeting Pipeline
This cell implements the main pipeline for Iteration 5, including aggressive class balancing (SMOTE, undersampling), embedding generation, classifier training, threshold optimization, and evaluation. Run this cell to process datasets, train models, and generate results and visualizations for process safety classification.

In [ ]:
# ============================================================
#  ITERATION 5 - AGGRESSIVE PROCESS SAFETY TARGETING
# ============================================================
# Goal: Achieve 98% Macro F1-score through:
# 1. Full SMOTE oversampling (auto/1.0) for Process Safety
# 2. Strategic undersampling of Non-Process Safety
# 3. Optimized cost-sensitive learning
# 4. Threshold optimization for PS recall
# ============================================================

import pandas as pd
import numpy as np
import os
import re
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import pickle
import glob
import gc
import json
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available. Install with: pip install xgboost")
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    print("LightGBM not available. Install with: pip install lightgbm")
    LIGHTGBM_AVAILABLE = False

try:
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours
    from imblearn.combine import SMOTETomek, SMOTEENN
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("imbalanced-learn not available. Install with: pip install imbalanced-learn")
    IMBLEARN_AVAILABLE = False

print("\n" + "#"*80)
print("ITERATION 5: AGGRESSIVE PROCESS SAFETY TARGETING")
print("Target: 98% Macro F1-Score")
print("Strategy: Full SMOTE Oversampling + Strategic Undersampling")
print("#"*80)

# ============================================================
# CONFIGURATION - Cross-platform compatible paths
# ============================================================

# Use paths from Cell 1 if available, otherwise define them
# Paths are defined in Cell 1 via find_project_root_with_datasets()

print(f"\nUsing paths:")
print(f"   DATA_DIR: {DATA_DIR}")
print(f"   EMBEDDINGS_BASE_DIR: {EMBEDDINGS_BASE_DIR}")
print(f"   RESULTS_DIR: {RESULTS_DIR}")
print(f"   CHECKPOINT_FILE: {CHECKPOINT_FILE}")

os.makedirs(EMBEDDINGS_BASE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)

# ITERATION 5 ENHANCED CONFIGURATION
CONFIG = {
    # Embedding strategy
    'pooling_strategy': 'mean',
    'text_preprocessing': True,
    'use_feature_scaling': True,
    
    # AGGRESSIVE CLASS BALANCING (Key Changes)
    'use_smote': True,
    'smote_strategy': 'auto',  # Full 1:1 balancing (was 0.7 in _iteration_4)
    'use_undersampling': True,  # NEW: Undersample majority class
    'undersample_strategy': 0.8,  # Target ratio: majority = 0.8 * minority
    'combine_smote_tomek': True,  # Clean overlapping samples
    
    # Classifier optimization
    'hyperparameter_tuning': True,
    'use_ensemble': True,
    'classifiers': ['svm', 'xgboost', 'lightgbm', 'random_forest', 'ensemble'],
    'cv_folds': 5,
    
    # NEW: Threshold optimization for PS recall
    'optimize_threshold': True,
    'target_ps_recall': 0.95,  # Aim for 95% PS recall
    
    # NEW: Aggressive class weights
    'aggressive_class_weights': True,
    'ps_weight_multiplier': 10,  # 10x weight on PS incidents
}

# Best performing models from Iteration 4
MODELS = {
    'bert-base-multilingual-uncased': 'google-bert/bert-base-multilingual-uncased',
    'xlm-roberta-base': 'xlm-roberta-base', 
    'paraphrase-multilingual-mpnet-base-v2': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'paraphrase-multilingual-MiniLM-L12-v2': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'snowflake-arctic-embed-l-v2.0': 'snowflake/snowflake-arctic-embed-l-v2.0',
    'labse-gguf': 'ChristianAzinn/labse-gguf',
    'upskyy/bge-m3-korean': 'upskyy/bge-m3-korean',
    'intfloat/multilingual-e5-large-instruct': 'intfloat/multilingual-e5-large-instruct',
    'efederici/e5-base-multilingual-4096': 'efederici/e5-base-multilingual-4096',
}

SKIP_DATASETS = []  # Not used in the JSON-based Iteration 5 pipeline

BATCH_SIZE = 12
MAX_SAMPLES_SVM = 6000 

print(f"\nITERATION 5 CONFIGURATION:")
print(f"   |- Pooling Strategy: {CONFIG['pooling_strategy']}")
print(f"   |- SMOTE Strategy: {CONFIG['smote_strategy']} (Full 1:1 Balancing)")
print(f"   |- Undersampling: {CONFIG['use_undersampling']} (Strategy: {CONFIG['undersample_strategy']})")
print(f"   |- SMOTE+Tomek: {CONFIG['combine_smote_tomek']}")
print(f"   |- Threshold Optimization: {CONFIG['optimize_threshold']}")
print(f"   |- Aggressive Class Weights: {CONFIG['aggressive_class_weights']} (PS Multiplier: {CONFIG['ps_weight_multiplier']}x)")
print(f"   |- Models: {len(MODELS)}")
print(f"   |- Max SVM Samples: {MAX_SAMPLES_SVM:,}")

# ============================================================
# TEXT PREPROCESSING
# ============================================================

def advanced_text_preprocessing(text):
    """Enhanced text preprocessing"""
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s\.]', ' ', text)
    text = ' '.join(text.split())
    
    return text

# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def save_checkpoint(processed_items):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'processed': list(processed_items)}, f)
    print(f"Checkpoint saved: {len(processed_items)} combinations processed")

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return set(tuple(item) for item in json.load(f)['processed'])
    return set()

def reset_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared")
    else:
        print("No checkpoint found")

# ============================================================
# EMBEDDING GENERATION
# ============================================================

def get_embeddings(texts, model_name, model_path, dataset_name, batch_size=12, pooling='mean'):
    """Generate embeddings using transformer model with specified pooling"""
    
    # Include dataset name in cache filename to avoid loading wrong embeddings
    safe_dataset_name = os.path.splitext(dataset_name)[0].replace('/', '_')
    embedding_file = os.path.join(
        EMBEDDINGS_BASE_DIR, 
        f"embeddings_{model_name.replace('/', '_')}_{safe_dataset_name}_{pooling}.npy"
    )
    labels_file = os.path.join(
        EMBEDDINGS_BASE_DIR,
        f"labels_{model_name.replace('/', '_')}_{safe_dataset_name}.npy"
    )
    
    if os.path.exists(embedding_file):
        print(f"Loading cached embeddings from {embedding_file}")
        return np.load(embedding_file)
    
    print(f"Generating embeddings with {model_name} using {pooling} pooling...")
    
    device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path).to(device)
    model.eval()
    
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i+batch_size].tolist()
        
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            
            if pooling == 'cls':
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            elif pooling == 'mean':
                attention_mask = encoded['attention_mask']
                token_embeddings = outputs.last_hidden_state
                input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
                sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
                sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                batch_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
            elif pooling == 'max':
                token_embeddings = outputs.last_hidden_state
                batch_embeddings = torch.max(token_embeddings, dim=1)[0].cpu().numpy()
            else:
                raise ValueError(f"Unknown pooling strategy: {pooling}")
            
            embeddings.extend(batch_embeddings)
        
        if device == 'cuda':
            torch.cuda.empty_cache()
        if i % 50 == 0 and i > 0:
            gc.collect()
    
    embeddings = np.array(embeddings)
    np.save(embedding_file, embeddings)
    print(f"Embeddings saved to {embedding_file}")
    
    return embeddings

# ============================================================
# AGGRESSIVE CLASS BALANCING PIPELINE
# ============================================================

def apply_aggressive_balancing(X_train, y_train):
    """
    Apply aggressive class balancing:
    1. SMOTE oversampling to 1:1 ratio (auto)
    2. Optional: Clean overlapping samples with Tomek Links
    3. Strategic undersampling of majority class
    """
    
    if not IMBLEARN_AVAILABLE:
        print("imbalanced-learn not available. Skipping balancing.")
        return X_train, y_train, False, {}
    
    print(f"\nAGGRESSIVE CLASS BALANCING PIPELINE")
    print(f"{'─'*60}")
    
    original_counts = np.bincount(y_train)
    print(f"Original class distribution:")
    print(f"   Non-PS (0): {original_counts[0]:,} ({original_counts[0]/len(y_train)*100:.1f}%)")
    print(f"   PS (1):     {original_counts[1]:,} ({original_counts[1]/len(y_train)*100:.1f}%)")
    print(f"   Imbalance ratio: {original_counts[0]/original_counts[1]:.2f}:1")
    
    try:
        # STEP 1: SMOTE Oversampling (Full 1:1 balancing)
        print(f"\nSTEP 1: SMOTE Oversampling (strategy='{CONFIG['smote_strategy']}')")
        smote = SMOTE(
            sampling_strategy=CONFIG['smote_strategy'],  # 'auto' = 1:1 ratio
            random_state=42,
            k_neighbors=min(5, sum(y_train == 1) - 1)
        )
        X_balanced, y_balanced = smote.fit_resample(X_train, y_train)
        
        smote_counts = np.bincount(y_balanced)
        print(f"   After SMOTE:")
        print(f"      Non-PS: {smote_counts[0]:,}")
        print(f"      PS:     {smote_counts[1]:,}")
        print(f"      New ratio: {smote_counts[0]/smote_counts[1]:.2f}:1")
        
        # STEP 2: Clean overlapping samples (Optional)
        if CONFIG['combine_smote_tomek']:
            print(f"\nSTEP 2: Tomek Links Cleaning (remove overlapping samples)")
            tomek = TomekLinks(sampling_strategy='majority')
            X_balanced, y_balanced = tomek.fit_resample(X_balanced, y_balanced)
            
            tomek_counts = np.bincount(y_balanced)
            print(f"   After Tomek Links:")
            print(f"      Non-PS: {tomek_counts[0]:,} (removed {smote_counts[0] - tomek_counts[0]:,})")
            print(f"      PS:     {tomek_counts[1]:,} (removed {smote_counts[1] - tomek_counts[1]:,})")
        
        # STEP 3: Strategic Undersampling of Majority Class
        if CONFIG['use_undersampling']:
            print(f"\nSTEP 3: Random Undersampling (strategy={CONFIG['undersample_strategy']})")
            
            current_counts = np.bincount(y_balanced)
            majority_count = current_counts[0]
            minority_count = current_counts[1]
            
            # Strategy means target ratio: majority = minority * undersample_strategy
            # So 0.8 means majority will be 80% of minority count
            target_majority = int(minority_count * CONFIG['undersample_strategy'])
            
            # Only undersample if target is less than current count
            if target_majority < majority_count:
                undersampler = RandomUnderSampler(
                    sampling_strategy={0: target_majority, 1: minority_count},
                    random_state=42
                )
                X_balanced, y_balanced = undersampler.fit_resample(X_balanced, y_balanced)
                
                final_counts = np.bincount(y_balanced)
                print(f"   After Undersampling:")
                print(f"      Non-PS: {final_counts[0]:,}")
                print(f"      PS:     {final_counts[1]:,}")
                print(f"      Final ratio: {final_counts[0]/final_counts[1]:.2f}:1")
            else:
                print(f"   Skipping undersampling - target ({target_majority:,}) >= current majority ({majority_count:,})")
                print(f"   Current class balance is already achieved")
        
        # Summary
        final_counts = np.bincount(y_balanced)
        print(f"\nBALANCING SUMMARY:")
        print(f"   Original size: {len(y_train):,} -> Final size: {len(y_balanced):,}")
        print(f"   Original imbalance: {original_counts[0]/original_counts[1]:.2f}:1")
        print(f"   Final imbalance: {final_counts[0]/final_counts[1]:.2f}:1")
        print(f"   PS samples gained: {final_counts[1] - original_counts[1]:,} (+{(final_counts[1]/original_counts[1]-1)*100:.1f}%)")
        
        metadata = {
            'original_counts': original_counts.tolist(),
            'final_counts': final_counts.tolist(),
            'smote_strategy': CONFIG['smote_strategy'],
            'undersampling': CONFIG['use_undersampling'],
            'undersample_strategy': CONFIG['undersample_strategy'] if CONFIG['use_undersampling'] else None,
            'tomek_cleaning': CONFIG['combine_smote_tomek'],
        }
        
        return X_balanced, y_balanced, True, metadata
        
    except Exception as e:
        print(f"\nBalancing failed: {e}")
        print(f"   Using original training data")
        import traceback
        traceback.print_exc()
        return X_train, y_train, False, {}

# ============================================================
# THRESHOLD OPTIMIZATION FOR RECALL
# ============================================================

def optimize_threshold_for_ps_recall(y_true, y_proba, target_recall=0.95):
    """
    Find optimal classification threshold to achieve target PS recall
    while maximizing F1-score
    """
    
    if y_proba is None:
        return None, None
    
    print(f"\nOPTIMIZING THRESHOLD FOR PS RECALL >= {target_recall}")
    print(f"{'─'*60}")
    
    thresholds = np.arange(0.1, 0.91, 0.05)
    best_threshold = 0.5
    best_f1 = 0
    best_recall = 0
    
    results = []
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=[0, 1]
        )
        
        ps_recall = recall[1]
        ps_f1 = f1[1]
        macro_f1 = np.mean(f1)
        
        results.append({
            'threshold': threshold,
            'ps_recall': ps_recall,
            'ps_f1': ps_f1,
            'macro_f1': macro_f1
        })
        
        # Find best threshold that meets recall target
        if ps_recall >= target_recall and macro_f1 > best_f1:
            best_threshold = threshold
            best_f1 = macro_f1
            best_recall = ps_recall
    
    print(f"Threshold optimization results:")
    for r in results[::2]:  # Print every other result
        marker = " <-- BEST" if r['threshold'] == best_threshold else ""
        print(f"   Threshold {r['threshold']:.2f}: PS Recall={r['ps_recall']:.3f}, "
              f"PS F1={r['ps_f1']:.3f}, Macro F1={r['macro_f1']:.3f}{marker}")
    
    print(f"\nOptimal Threshold: {best_threshold:.2f}")
    print(f"   PS Recall: {best_recall:.3f} (target: {target_recall})")
    print(f"   Macro F1: {best_f1:.3f}")
    
    return best_threshold, results

# ============================================================
# CLASSIFIER TRAINING WITH AGGRESSIVE SETTINGS
# ============================================================

def train_classifiers(X_train, y_train, X_test, y_test):
    """Train multiple classifiers with aggressive PS-focused settings"""
    
    results = {}
    
    # Calculate aggressive class weights
    n_ps = sum(y_train == 1)
    n_nps = sum(y_train == 0)
    
    if CONFIG['aggressive_class_weights']:
        # PS incidents get 10x weight multiplier
        weight_nps = 1.0
        weight_ps = (n_nps / n_ps) * CONFIG['ps_weight_multiplier']
        class_weights = {0: weight_nps, 1: weight_ps}
        print(f"\nAggressive Class Weights: NPS={weight_nps:.2f}, PS={weight_ps:.2f}")
    else:
        class_weights = 'balanced'
    
    # SVM
    print("\n" + "="*60)
    print("Training SVM with aggressive settings...")
    print("="*60)
    
    svm_params = {
        'C': [0.1, 1, 10, 100],
        'class_weight': [class_weights, 'balanced']
    }
    
    svm = GridSearchCV(
        LinearSVC(dual=False, max_iter=5000),
        svm_params,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    svm.fit(X_train, y_train)
    
    y_pred_svm = svm.predict(X_test)
    results['SVM'] = {
        'model': svm.best_estimator_,
        'y_pred': y_pred_svm,
        'best_params': svm.best_params_,
        'cv_score': svm.best_score_
    }
    
    print(f"Best SVM params: {svm.best_params_}")
    print(f"CV F1 Score: {svm.best_score_:.4f}")
    
    # XGBoost
    if XGBOOST_AVAILABLE:
        print("\n" + "="*60)
        print("Training XGBoost with aggressive settings...")
        print("="*60)
        
        scale_pos_weight = n_nps / n_ps * CONFIG['ps_weight_multiplier'] if CONFIG['aggressive_class_weights'] else n_nps / n_ps
        
        xgb = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False
        )
        xgb.fit(X_train, y_train)
        
        y_pred_xgb = xgb.predict(X_test)
        y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
        
        # Threshold optimization
        if CONFIG['optimize_threshold']:
            opt_threshold, _ = optimize_threshold_for_ps_recall(
                y_test, y_proba_xgb, CONFIG['target_ps_recall']
            )
            y_pred_xgb_opt = (y_proba_xgb >= opt_threshold).astype(int)
            results['XGBoost_Optimized'] = {
                'model': xgb,
                'y_pred': y_pred_xgb_opt,
                'y_proba': y_proba_xgb,
                'threshold': opt_threshold
            }
        
        results['XGBoost'] = {
            'model': xgb,
            'y_pred': y_pred_xgb,
            'y_proba': y_proba_xgb
        }
    
    # LightGBM
    if LIGHTGBM_AVAILABLE:
        print("\n" + "="*60)
        print("Training LightGBM with aggressive settings...")
        print("="*60)
        
        lgbm = LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            class_weight=class_weights,
            random_state=42,
            verbose=-1
        )
        lgbm.fit(X_train, y_train)
        
        y_pred_lgbm = lgbm.predict(X_test)
        y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
        
        if CONFIG['optimize_threshold']:
            opt_threshold, _ = optimize_threshold_for_ps_recall(
                y_test, y_proba_lgbm, CONFIG['target_ps_recall']
            )
            y_pred_lgbm_opt = (y_proba_lgbm >= opt_threshold).astype(int)
            results['LightGBM_Optimized'] = {
                'model': lgbm,
                'y_pred': y_pred_lgbm_opt,
                'y_proba': y_proba_lgbm,
                'threshold': opt_threshold
            }
        
        results['LightGBM'] = {
            'model': lgbm,
            'y_pred': y_pred_lgbm,
            'y_proba': y_proba_lgbm
        }
    
    # Random Forest
    print("\n" + "="*60)
    print("Training Random Forest with aggressive settings...")
    print("="*60)
    
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        class_weight=class_weights,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    
    y_pred_rf = rf.predict(X_test)
    y_proba_rf = rf.predict_proba(X_test)[:, 1]
    
    if CONFIG['optimize_threshold']:
        opt_threshold, _ = optimize_threshold_for_ps_recall(
            y_test, y_proba_rf, CONFIG['target_ps_recall']
        )
        y_pred_rf_opt = (y_proba_rf >= opt_threshold).astype(int)
        results['Random_Forest_Optimized'] = {
            'model': rf,
            'y_pred': y_pred_rf_opt,
            'y_proba': y_proba_rf,
            'threshold': opt_threshold
        }
    
    results['Random_Forest'] = {
        'model': rf,
        'y_pred': y_pred_rf,
        'y_proba': y_proba_rf
    }
    
    # Voting Ensemble
    print("\n" + "="*60)
    print("Training Voting Ensemble...")
    print("="*60)
    
    estimators = [('rf', rf)]
    if XGBOOST_AVAILABLE:
        estimators.append(('xgb', xgb))
    if LIGHTGBM_AVAILABLE:
        estimators.append(('lgbm', lgbm))
    
    if len(estimators) > 1:
        ensemble = VotingClassifier(
            estimators=estimators,
            voting='soft'
        )
        ensemble.fit(X_train, y_train)
        
        y_pred_ens = ensemble.predict(X_test)
        y_proba_ens = ensemble.predict_proba(X_test)[:, 1]
        
        results['Voting_Ensemble'] = {
            'model': ensemble,
            'y_pred': y_pred_ens,
            'y_proba': y_proba_ens
        }
    
    return results

# ============================================================
# EVALUATION AND VISUALIZATION
# ============================================================

def evaluate_and_visualize(results, y_test, dataset_name, model_name):
    """Comprehensive evaluation with visualization"""
    
    evaluation_results = []
    
    for clf_name, clf_data in results.items():
        y_pred = clf_data['y_pred']
        
        cm = confusion_matrix(y_test, y_pred)
        TN, FP, FN, TP = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
        
        accuracy = (TP + TN) / (TP + TN + FP + FN)
        precision_ps = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall_ps = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1_ps = 2 * (precision_ps * recall_ps) / (precision_ps + recall_ps) if (precision_ps + recall_ps) > 0 else 0
        
        precision_nps = TN / (TN + FN) if (TN + FN) > 0 else 0
        recall_nps = TN / (TN + FP) if (TN + FP) > 0 else 0
        f1_nps = 2 * (precision_nps * recall_nps) / (precision_nps + recall_nps) if (precision_nps + recall_nps) > 0 else 0
        
        macro_f1 = (f1_ps + f1_nps) / 2
        
        eval_result = {
            'dataset': dataset_name,
            'embedding_model': model_name,
            'classifier': clf_name,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'precision_ps': precision_ps,
            'recall_ps': recall_ps,
            'f1_ps': f1_ps,
            'precision_nps': precision_nps,
            'recall_nps': recall_nps,
            'f1_nps': f1_nps,
            'TP': TP,
            'TN': TN,
            'FP': FP,
            'FN': FN,
            'threshold': clf_data.get('threshold', 0.5)
        }
        evaluation_results.append(eval_result)
        
        # Print results
        print(f"\n{'='*60}")
        print(f"{clf_name} Results:")
        print(f"{'='*60}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(f"\nProcess Safety (PS):")
        print(f"   Precision: {precision_ps:.4f}")
        print(f"   Recall: {recall_ps:.4f}")
        print(f"   F1: {f1_ps:.4f}")
        print(f"\nNon-Process Safety (NPS):")
        print(f"   Precision: {precision_nps:.4f}")
        print(f"   Recall: {recall_nps:.4f}")
        print(f"   F1: {f1_nps:.4f}")
        print(f"\nConfusion Matrix:")
        print(f"   TN={TN}, FP={FP}")
        print(f"   FN={FN}, TP={TP}")
        
        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Confusion Matrix
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                    xticklabels=['Non-PS', 'PS'], yticklabels=['Non-PS', 'PS'])
        axes[0].set_title(f'{clf_name} Confusion Matrix\n{dataset_name} - {model_name}')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Actual')
        
        # Metrics Bar Chart
        metrics = ['Precision PS', 'Recall PS', 'F1 PS', 'Precision NPS', 'Recall NPS', 'F1 NPS', 'Macro F1']
        values = [precision_ps, recall_ps, f1_ps, precision_nps, recall_nps, f1_nps, macro_f1]
        colors = ['#2ecc71', '#2ecc71', '#2ecc71', '#3498db', '#3498db', '#3498db', '#e74c3c']
        
        bars = axes[1].bar(metrics, values, color=colors)
        axes[1].set_ylim(0, 1)
        axes[1].set_ylabel('Score')
        axes[1].set_title(f'{clf_name} Metrics')
        axes[1].tick_params(axis='x', rotation=45)
        
        for bar, val in zip(bars, values):
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        
        # Save figure
        fig_path = os.path.join(RESULTS_DIR, f"eval_{dataset_name}_{model_name.replace('/', '_')}_{clf_name}.png")
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"Visualization saved: {fig_path}")
        plt.close()
    
    return evaluation_results

# ============================================================
# MAIN PROCESSING PIPELINE
# ============================================================

def process_iteration_5():
    """Main processing function for Iteration 5"""
    
    print("\n" + "#"*80)
    print("STARTING ITERATION 5 PROCESSING")
    print("#"*80)
    
    all_results = []
    processed = load_checkpoint()
    
    # Get dataset files from the Iteration 0 JSON outputs
    dataset_items = list(DATASET_FILES.items())
    print(f"\nDatasets to process: {[name for name, _ in dataset_items]}")
    print(f"Models to use: {list(MODELS.keys())}")
    print(f"Already processed: {len(processed)} combinations")
    if not dataset_items:
        raise FileNotFoundError("DATASET_FILES is empty. Run the configuration cell first and verify the Iteration 0 JSON outputs exist.")
    
    for model_name, model_path in MODELS.items():
        for dataset_key, dataset_path in dataset_items:
            
            combo_key = (model_name, dataset_key)
            if combo_key in processed:
                print(f"\nSkipping already processed: {model_name} + {dataset_key}")
                continue
            
            print(f"\n{'#'*80}")
            print(f"Processing: {model_name} + {dataset_key}")
            print(f"{'#'*80}")
            
            try:
                # Load data
                df = pd.read_json(dataset_path)
                
                required_columns = {'TITLE', 'CASE_DESCRIPTION', 'CASE_TYPE'}
                missing_columns = required_columns - set(df.columns)
                if missing_columns:
                    raise KeyError(f"Missing required columns in {dataset_key}: {sorted(missing_columns)}")

                # Create binary label from CASE_TYPE column
                df['binary_label'] = df['CASE_TYPE'].apply(lambda x: 1 if x == 'Process Safety' else 0)

                # Create text column from TITLE and CASE_DESCRIPTION
                title_text = df['TITLE'].fillna('').astype(str)
                desc_text = df['CASE_DESCRIPTION'].fillna('').astype(str)
                if CONFIG['text_preprocessing']:
                    df['processed_text'] = [
                        advanced_text_preprocessing(f"{title}. {desc}")
                        for title, desc in zip(title_text, desc_text)
                    ]
                else:
                    df['processed_text'] = title_text + '. ' + desc_text
                
                print(f"Dataset size: {len(df)}")
                print(f"Class distribution: {df['binary_label'].value_counts().to_dict()}")
                
                # Get embeddings - now includes dataset_file for proper caching
                embeddings = get_embeddings(
                    df['processed_text'],
                    model_name,
                    model_path,
                    dataset_key,  # Use dataset key for cache-safe embedding filenames
                    batch_size=BATCH_SIZE,
                    pooling=CONFIG['pooling_strategy']
                )
                
                labels = df['binary_label'].values
                
                # Train/test split
                X_train, X_test, y_train, y_test = train_test_split(
                    embeddings, labels, test_size=0.2, random_state=42, stratify=labels
                )
                
                print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
                
                # Scale features
                if CONFIG['use_feature_scaling']:
                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(X_train)
                    X_test = scaler.transform(X_test)
                
                # Apply aggressive balancing
                X_train_balanced, y_train_balanced, balanced, balance_meta = apply_aggressive_balancing(
                    X_train, y_train
                )
                
                # Train classifiers
                results = train_classifiers(
                    X_train_balanced, y_train_balanced,
                    X_test, y_test
                )
                
                # Evaluate
                dataset_name = dataset_key
                eval_results = evaluate_and_visualize(
                    results, y_test, dataset_name, model_name
                )
                
                all_results.extend(eval_results)
                
                # Update checkpoint
                processed.add(combo_key)
                save_checkpoint(processed)
                
                # Save intermediate results
                results_df = pd.DataFrame(all_results)
                results_df.to_json(
                    os.path.join(RESULTS_DIR, 'iteration_5_comprehensive_results.json'),
                    orient='records',
                    force_ascii=False,
                    indent=2
                )
                results_df.to_csv(
                    os.path.join(RESULTS_DIR, 'iteration_5_comprehensive_results.csv'),
                    index=False
                )
                
                # Clean up
                del embeddings, X_train, X_test, y_train, y_test
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"Error processing {model_name} + {dataset_key}: {e}")
                import traceback
                traceback.print_exc()
                continue
    
    # Final summary
    print("\n" + "#"*80)
    print("ITERATION 5 COMPLETE")
    print("#"*80)
    
    if all_results:
        results_df = pd.DataFrame(all_results)
        
        # Save final results
        final_json_path = os.path.join(RESULTS_DIR, 'iteration_5_comprehensive_results.json')
        final_csv_path = os.path.join(RESULTS_DIR, 'iteration_5_comprehensive_results.csv')
        results_df.to_json(final_json_path, orient='records', force_ascii=False, indent=2)
        results_df.to_csv(final_csv_path, index=False)
        print(f"\nResults saved to: {final_json_path}")
        print(f"Compatibility CSV saved to: {final_csv_path}")
        
        # Print summary
        print(f"\nTotal evaluations: {len(results_df)}")
        print(f"\nTop 10 by Macro F1:")
        top_10 = results_df.nlargest(10, 'macro_f1')[
            ['embedding_model', 'classifier', 'dataset', 'macro_f1', 'recall_ps', 'f1_ps']
        ]
        print(top_10.to_string(index=False))
        
        # Best overall
        best = results_df.loc[results_df['macro_f1'].idxmax()]
        print(f"\nBEST OVERALL:")
        print(f"   Model: {best['embedding_model']}")
        print(f"   Classifier: {best['classifier']}")
        print(f"   Dataset: {best['dataset']}")
        print(f"   Macro F1: {best['macro_f1']:.4f}")
        print(f"   PS Recall: {best['recall_ps']:.4f}")
        print(f"   PS F1: {best['f1_ps']:.4f}")
    
    return all_results

# Run the processing
if __name__ == "__main__":
    process_iteration_5()

# =============================================================================
# Save iteration_5_summary.json (consistent with Iteration 0 format)
# =============================================================================
import json as _json5, pandas as _pd5

_results_json5 = os.path.join(RESULTS_DIR, 'iteration_5_comprehensive_results.json')
_results_csv5 = os.path.join(RESULTS_DIR, 'iteration_5_comprehensive_results.csv')
if os.path.exists(_results_json5):
    _df5 = _pd5.read_json(_results_json5)
elif os.path.exists(_results_csv5):
    _df5 = _pd5.read_csv(_results_csv5)
    _best5 = _df5.loc[_df5['macro_f1'].idxmax()] if 'macro_f1' in _df5.columns else _df5.iloc[0]
    _summary5 = {
        'iteration': 5,
        'model': '9 multilingual embedding models + SMOTE balancing + classical ML classifiers',
        'strategy': 'SMOTE oversampling + aggressive class weighting',
        'best_result': {
            'embedding_model': str(_best5.get('embedding_model', '')),
            'classifier': str(_best5.get('classifier', '')),
            'dataset': str(_best5.get('dataset', '')),
            'macro_f1': round(float(_best5.get('macro_f1', 0)), 4),
            'recall_ps': round(float(_best5.get('recall_ps', 0)), 4),
        },
        'gap_analysis': {
            'current_macro_f1': round(float(_best5.get('macro_f1', 0)), 4),
            'rq1_target': 0.8161,
            'gap': round(float(0.8161 - _best5.get('macro_f1', 0)), 4)
        }
    }
    with open(os.path.join(RESULTS_DIR, 'iteration_5_summary.json'), 'w') as _f5:
        _json5.dump(_summary5, _f5, indent=2)
    print(f"[OK] Saved: iteration_5_summary.json")
else:
    print(f"[INFO] Results CSV not found yet. Run pipeline first.")


/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



################################################################################
ITERATION 5: AGGRESSIVE PROCESS SAFETY TARGETING
Target: 98% Macro F1-Score
Strategy: Full SMOTE Oversampling + Strategic Undersampling
################################################################################

Using paths:
   DATA_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/By_SL_Country
   EMBEDDINGS_BASE_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Embeddings/_iteration_5
   RESULTS_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_5
   CHECKPOINT_FILE: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_5/checkpoint_iteration_5.json

ITERATION 5 CONFIGURATION:
   |- Pooling Strategy: mean
   |- SMOTE Strategy: auto (Full 1:1 Balancing)
   |- Undersampling: True (Strategy: 0.8)
   |- SMOTE+Tomek: True
   |- Threshold Optimization: True
   |- Aggressive Class Weights: True (PS Mult

Generating embeddings: 100%|██████████| 393/393 [01:20<00:00,  4.86it/s]


Embeddings saved to /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Embeddings/_iteration_5/embeddings_bert-base-multilingual-uncased_english_manual_mean.npy
Train size: 3770, Test size: 943

AGGRESSIVE CLASS BALANCING PIPELINE
────────────────────────────────────────────────────────────
Original class distribution:
   Non-PS (0): 2,643 (70.1%)
   PS (1):     1,127 (29.9%)
   Imbalance ratio: 2.35:1

STEP 1: SMOTE Oversampling (strategy='auto')
   After SMOTE:
      Non-PS: 2,643
      PS:     2,643
      New ratio: 1.00:1

STEP 2: Tomek Links Cleaning (remove overlapping samples)
   After Tomek Links:
      Non-PS: 2,642 (removed 1)
      PS:     2,643 (removed 0)

STEP 3: Random Undersampling (strategy=0.8)
   After Undersampling:
      Non-PS: 2,114
      PS:     2,643
      Final ratio: 0.80:1

BALANCING SUMMARY:
   Original size: 3,770 -> Final size: 4,757
   Original imbalance: 2.35:1
   Final imbalance: 0.80:1
   PS samples gained: 1,516 (+134.5%)

Aggressive Class We

Generating embeddings: 100%|██████████| 1246/1246 [03:00<00:00,  6.90it/s]


Embeddings saved to /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Embeddings/_iteration_5/embeddings_bert-base-multilingual-uncased_german_manual_mean.npy
Train size: 11960, Test size: 2991

AGGRESSIVE CLASS BALANCING PIPELINE
────────────────────────────────────────────────────────────
Original class distribution:
   Non-PS (0): 10,631 (88.9%)
   PS (1):     1,329 (11.1%)
   Imbalance ratio: 8.00:1

STEP 1: SMOTE Oversampling (strategy='auto')
   After SMOTE:
      Non-PS: 10,631
      PS:     10,631
      New ratio: 1.00:1

STEP 2: Tomek Links Cleaning (remove overlapping samples)
   After Tomek Links:
      Non-PS: 10,631 (removed 0)
      PS:     10,631 (removed 0)

STEP 3: Random Undersampling (strategy=0.8)
   After Undersampling:
      Non-PS: 8,504
      PS:     10,631
      Final ratio: 0.80:1

BALANCING SUMMARY:
   Original size: 11,960 -> Final size: 19,135
   Original imbalance: 8.00:1
   Final imbalance: 0.80:1
   PS samples gained: 9,302 (+699.9%)

Aggressive

Generating embeddings:  14%|█▍        | 147/1019 [00:10<00:55, 15.64it/s]